<a href="https://colab.research.google.com/github/Wapikkk/text-summarize-nlp/blob/abstractive/notebooks/03_abstractive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Abstractive Summarization (WikiLingua, Bahasa Indonesia)

**Tujuan**: membuat ringkasan dengan menulis ulang isi teks (bukan memilih kalimat) dan membandingkannya dengan metode extractive (02) di evaluasi (04).

Isi notebook:
1. Setup dan konfigurasi
2. Memuat split resmi dari peran 1 (`train/val/test.jsonl`, satuan data = satu bagian artikel)
3. Baseline **Seq2Seq GRU + attention** (dilatih dari nol)
4. **Fine-tune model pretrained** (default `google/mt5-small`)
5. Menyimpan hasil ke JSONL dengan format kontrak tim: `{"id", "reference", "prediction"}`

**Cara pakai:** run sel dari atas ke bawah dengan runtime **GPU (T4)**. Untuk uji coba cepat, set `DEBUG = True` di sel konfigurasi (data kecil hasil akan otomatis langsung ke `scratch/` di google drive). Apabila session timeout, run ulang dari atas, checkpoint yang sudah tersimpan di Drive otomatis dipakai kembali.

## **Install Package & Import Package**

Memasang library (`transformers` + `sentencepiece` untuk mT5, `rouge-score` untuk cek cepat), mengimpor modul, menggunakan seed (42), dan checking apakah GPU aktif. Penggunaan GPU didasarkan karena banyaknya pelatihan dengan neuron jutaan parameter.

In [ ]:
!pip install -q transformers sentencepiece accelerate rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os, re, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from collections import Counter
from google.colab import drive

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup

from rouge_score import rouge_scorer

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cpu":
    print("Switch runtime dengan GPU T4. Pilih change runtime type, lalu T4 GPU.")

Device: cuda


## **Konfigurasi**
Semua pengaturan ada di sel ini supaya mudah dicatat di laporan.
- `DEBUG`: `True` untuk uji coba cepat (data dipotong, 1 epoch, hasil dimasukkan `scratch/`). Ubah ke `False` untuk running semua dokumen.
- `RETRAIN`: `False` menandakan bahwa checkpoint yang sudah ada di Drive dipakai lagi. `True` untuk training ulang dari awal.
- Panjang maksimum dipilih dari statistik data (median dokumen 304 kata, median ringkasan 27 kata): dokumen yang lebih panjang dipotong, dan hal tersebut akan dicatat sebagai keterbatasan.

In [ ]:
# Mode
DEBUG = False       # True = uji coba sebagian dokumen atau False = run semua dokumen
RETRAIN = False    # False = pakai checkpoint yang sudah ada di Drive

# Lokasi (Google Drive bersama)
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/dokumen_nlp"   # folder berisi train.jsonl, val.jsonl, test.jsonl dari hasil preprocessing 01_data
SPLIT_DIR = DRIVE_DIR
OUT_DIR = f"{DRIVE_DIR}/scratch" if DEBUG else f"{DRIVE_DIR}/results"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints" + ("_debug" if DEBUG else "")
for d in (OUT_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print("DEBUG:", DEBUG, "| OUT_DIR:", OUT_DIR)

# Hyperparameter baseline GRU
LSTM_CFG = dict(
    max_src_words=300, max_tgt_words=60,
    vocab_size=30000, min_freq=3,
    emb=256, hid=256, dropout=0.3,
    epochs=10, batch_size=64, lr=1e-3, patience=3, tf_ratio=0.5,
)

# Hyperparameter model pretrained
PRETRAINED = "google/mt5-small"
PT_CFG = dict(
    max_src_tok=512, max_tgt_tok=96,
    epochs=3, batch_size=4, grad_accum=4, lr=3e-4,
    eval_batch_size=16, num_beams=4,
)

if DEBUG:
    LSTM_CFG.update(epochs=1)
    PT_CFG.update(epochs=1, max_src_tok=256)
    N_DEBUG = dict(train=800, val=100, test=100)

Mounted at /content/drive
DEBUG: False | OUT_DIR: /content/drive/MyDrive/dokumen_nlp/results


## **Load Data**
Membaca `train.jsonl`, `val.jsonl`, dan `test.jsonl` hasil 01_data (kolom `id`, `document`, `summary`, satuan = satu bagian artikel, split per artikel sehingga tidak ada leaking data). Train untuk melatih, val untuk memilih checkpoint terbaik, test hanya untuk hasil akhir.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = {k: read_jsonl(f"{SPLIT_DIR}/{k}.jsonl") for k in ("train", "val", "test")}
for k in data:
    data[k] = [r for r in data[k] if r.get("document") and r.get("summary")]
    if DEBUG:
        data[k] = data[k][:N_DEBUG[k]]

train_rows, val_rows, test_rows = data["train"], data["val"], data["test"]
print({k: len(v) for k, v in data.items()})
print("\nContoh:")
print("DOC :", train_rows[0]["document"][:300])
print("SUM :", train_rows[0]["summary"][:300])

{'train': 37970, 'val': 4725, 'test': 4778}

Contoh:
DOC : Jika tingkat pengembalian total telah dihitung, masukkan hasilnya ke persamaan berikut: Pengembalian Disetahunkan=(1+ Pengembalian)1/N-1 Hasil persamaan ini sesuai dengan pengembalian investasi setiap tahunnya selama rentang waktu yang diukur. Di dalam eksponen (pangkat), angka “1” mencerminkan unit
SUM : Hitunglah pengembalian disetahunkan Anda. Hitunglah pengembalian paruh tahunan (semi-annual). Hitunglah ekuivalen disetahunkan.


Statistik panjang dokumen dan ringkasan untuk memeriksa apakah panjang maksimum di konfigurasi masuk akal. Nilai persentil ke-95 menunjukkan seberapa banyak teks yang terpotong.

In [ ]:
# Statistik untuk menentukan panjang maksimum (jangan diubah tanpa mencatatnya di laporan)
def wlen(t): return len(t.split())
dl = np.array([wlen(r["document"]) for r in train_rows])
sl = np.array([wlen(r["summary"]) for r in train_rows])
for name, a in (("document", dl), ("summary", sl)):
    print(f"{name}: rata-rata={a.mean():.1f} median={np.median(a):.0f} p95={np.percentile(a,95):.0f} maks={a.max()}")

document: rata-rata=346.3 median=304 p95=768 maks=2728
summary: rata-rata=30.8 median=27 p95=64 maks=265


## **Baseline: Seq2Seq GRU + attention**
Model encoder-decoder tanpa pre-training. Encoder = GRU dua arah membaca dokumen. Sedangkan, decoder = GRU dengan attention Bahdanau menulis ringkasan kata demi kata. Kosakata dibangun dari data train (kata di bawah `min_freq` menjadi `<unk>`), sehingga model ini rentan terhadap kata baru (OOV). Sel berikut mendefinisikan tokenisasi, kosakata, dan dataset.

In [ ]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3
TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class Vocab:
    def __init__(self, token_lists, max_size, min_freq):
        cnt = Counter(t for toks in token_lists for t in toks)
        words = [w for w, c in cnt.most_common() if c >= min_freq][: max_size - 4]
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def encode(self, toks):
        return [self.stoi.get(t, UNK) for t in toks]
    def __len__(self):
        return len(self.itos)

class SumDataset(Dataset):
    def __init__(self, rows, vocab, max_src, max_tgt):
        self.items = []
        for r in rows:
            s = vocab.encode(tokenize(r["document"])[:max_src]) or [UNK]
            t = [SOS] + vocab.encode(tokenize(r["summary"])[:max_tgt]) + [EOS]
            self.items.append((s, t))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def lstm_collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    pad = nn.utils.rnn.pad_sequence
    src = pad([torch.tensor(s) for s in srcs], batch_first=True, padding_value=PAD)
    tgt = pad([torch.tensor(t) for t in tgts], batch_first=True, padding_value=PAD)
    return src, src_len, tgt

**Arsitektur model.** `Encoder`, `Attention`, `Decoder`, dan `Seq2Seq` (teacher forcing acak sesuai `tf_ratio`), serta `greedy_decode` untuk kesimpulan berdasarkan bukti yang dihasilkan.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb, hid, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.rnn = nn.GRU(emb, hid, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid * 2, hid)
        self.drop = nn.Dropout(dropout)
    def forward(self, src, src_len):
        e = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(e, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        h = torch.tanh(self.fc(torch.cat([h[0], h[1]], dim=1))) # (B, hid)
        return out, h # out: (B, S, 2*hid)

class Attention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.W = nn.Linear(enc_dim + dec_dim, dec_dim)
        self.v = nn.Linear(dec_dim, 1, bias=False)
    def forward(self, h, enc_out, mask):
        S = enc_out.size(1)
        hh = h.unsqueeze(1).expand(-1, S, -1)
        energy = torch.tanh(self.W(torch.cat([hh, enc_out], dim=2)))
        scores = self.v(energy).squeeze(2).masked_fill(~mask, -1e9)
        return F.softmax(scores, dim=1) # (B, S)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb, enc_dim, hid, dropout):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=PAD)
        self.attn = Attention(enc_dim, hid)
        self.rnn = nn.GRU(emb + enc_dim, hid, batch_first=True)
        self.out = nn.Linear(hid + enc_dim + emb, vocab_size)
        self.drop = nn.Dropout(dropout)
    def forward(self, tok, h, enc_out, mask):
        e = self.drop(self.emb(tok)).unsqueeze(1) # (B,1,emb)
        a = self.attn(h, enc_out, mask).unsqueeze(1) # (B,1,S)
        ctx = torch.bmm(a, enc_out) # (B,1,enc)
        out, h_new = self.rnn(torch.cat([e, ctx], dim=2), h.unsqueeze(0))
        logits = self.out(torch.cat([out.squeeze(1), ctx.squeeze(1), e.squeeze(1)], dim=1))
        return logits, h_new.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb, hid, dropout):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb, hid, dropout)
        self.decoder = Decoder(vocab_size, emb, 2 * hid, hid, dropout)
    def forward(self, src, src_len, tgt, tf_ratio=0.5):
        enc_out, h = self.encoder(src, src_len)
        mask = src != PAD
        tok, outs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, h = self.decoder(tok, h, enc_out, mask)
            outs.append(logits)
            tok = tgt[:, t] if random.random() < tf_ratio else logits.argmax(1)
        return torch.stack(outs, dim=1) # (B, T-1, V)

@torch.no_grad()
def greedy_decode(model, src, src_len, max_len, min_len=6):
    model.eval()
    enc_out, h = model.encoder(src, src_len)
    mask = src != PAD
    tok = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    outs = []
    for t in range(max_len):
        logits, h = model.decoder(tok, h, enc_out, mask)
        logits[:, PAD] = -1e9
        logits[:, SOS] = -1e9
        if t < min_len:
            logits[:, EOS] = -1e9
        logits.scatter_(1, tok.unsqueeze(1), -1e9)
        tok = logits.argmax(1)
        outs.append(tok)
    return torch.stack(outs, dim=1).tolist()

**Kosakata dan DataLoader.** Bertujuan untuk membangun kosakata dari train, lalu membuat DataLoader untuk train/val/test. Jumlah parameter dihasilkan keperluan untuk laporan.

In [ ]:
cfg = LSTM_CFG
vocab = Vocab([tokenize(r["document"]) + tokenize(r["summary"]) for r in train_rows], cfg["vocab_size"], cfg["min_freq"])
print("Ukuran kosakata:", len(vocab))

def make_dl(rows, shuffle):
    ds = SumDataset(rows, vocab, cfg["max_src_words"], cfg["max_tgt_words"])
    return DataLoader(ds, batch_size=cfg["batch_size"], shuffle=shuffle, collate_fn=lstm_collate)

train_dl, val_dl = make_dl(train_rows, True), make_dl(val_rows, False)
test_dl = make_dl(test_rows, False) # urutan sama dengan test_rows (shuffle=False)

lstm = Seq2Seq(len(vocab), cfg["emb"], cfg["hid"], cfg["dropout"]).to(DEVICE)
print("Parameter:", sum(p.numel() for p in lstm.parameters()) / 1e6, "juta")

Ukuran kosakata: 30000
Parameter: 48.01592 juta


**Training GRU.** Loss cross-entropy, optimizer Adam, gradient clipping, dan early stopping berdasarkan val loss. Checkpoint terbaik disimpan di Drive. Apabala sudah ada dan `RETRAIN = False`, training dilewati.

In [ ]:
crit = nn.CrossEntropyLoss(ignore_index=PAD)

def run_epoch(model, dl, opt=None, tf_ratio=0.5):
    train = opt is not None
    model.train(train)
    tot, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for src, src_len, tgt in dl:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, tf_ratio)
            loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            if train:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tot += loss.item(); n += 1
    return tot / max(n, 1)

lstm_ckpt = f"{CKPT_DIR}/lstm_best.pt"
lstm_meta = f"{CKPT_DIR}/lstm_meta.json"

if os.path.exists(lstm_ckpt) and not RETRAIN:
    print("Checkpoint ditemukan, training dilewati:", lstm_ckpt)
    lstm_train_time = json.load(open(lstm_meta))["train_time_s"] if os.path.exists(lstm_meta) else None
else:
    opt = torch.optim.Adam(lstm.parameters(), lr=cfg["lr"])
    best, bad, history = 1e9, 0, []
    t0 = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        tr = run_epoch(lstm, train_dl, opt, cfg["tf_ratio"])
        va = run_epoch(lstm, val_dl, None, 1.0)
        history.append((ep, tr, va))
        print(f"Epoch {ep:02d} | train loss {tr:.4f} | val loss {va:.4f}")
        if va < best:
            best, bad = va, 0
            torch.save(lstm.state_dict(), lstm_ckpt)
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print("Early stopping.")
                break
    lstm_train_time = time.time() - t0
    json.dump({"train_time_s": round(lstm_train_time, 1), "history": history}, open(lstm_meta, "w"))
    print(f"Waktu training LSTM: {lstm_train_time:.1f} detik")
lstm.load_state_dict(torch.load(lstm_ckpt, map_location=DEVICE))

Checkpoint ditemukan, training dilewati: /content/drive/MyDrive/dokumen_nlp/checkpoints/lstm_best.pt


<All keys matched successfully>

**Inferensi dan penyimpanan hasil GRU.** Membuat ringkasan untuk seluruh data test dengan greedy decoding, mengukur waktunya, lalu menyimpan JSONL sesuai kontrak tim.

In [ ]:
def detok(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return re.sub(r"([.!?])(\s*[.!?])+", r"\1", text)

def lstm_predict(model, dl):
    preds = []
    for src, src_len, _ in dl:
        out = greedy_decode(model, src.to(DEVICE), src_len, cfg["max_tgt_words"])
        for seq in out:
            toks = []
            for i in seq:
                if i == EOS: break
                toks.append(vocab.itos[i])
            preds.append(detok(toks))
    return preds

def save_predictions(path, rows, preds):
    assert len(rows) == len(preds)
    with open(path, "w", encoding="utf-8") as f:
        for r, p in zip(rows, preds):
            f.write(json.dumps({"id": r["id"], "reference": r["summary"], "prediction": p}, ensure_ascii=False) + "\n")
    print("Disimpan:", path)

t0 = time.time()
lstm_preds = lstm_predict(lstm, test_dl)
lstm_infer_time = time.time() - t0
save_predictions(f"{OUT_DIR}/abstractive_lstm.jsonl", test_rows, lstm_preds)
print(f"Waktu inferensi LSTM: {lstm_infer_time:.1f} detik untuk {len(test_rows)} contoh")
for i in range(3):
    print("\nREF :", test_rows[i]["summary"][:200]); print("PRED:", lstm_preds[i][:200])

Disimpan: /content/drive/MyDrive/dokumen_nlp/results/abstractive_lstm.jsonl
Waktu inferensi LSTM: 21.9 detik untuk 4778 contoh

REF : Pikirkan topik yang ingin Anda mimpikan. Tidur selama delapan jam atau lebih tiap hari. Berbaring di kasur setelah bangun. Tulis atau catat mimpi Anda untuk meningkatkan peluang untuk mengingatnya. Me
PRED: lakukan diri anda....................................................

REF : Tidur tengkurap. Makan makanan tertentu sebelum tidur. Takut-takuti diri Anda sedikit sebelum tidur. Coba mengkonsumsi vitamin B6. Coba mengkonsumsi melatonin. Konsumsi zat tertentu dalam porsi yang t
PRED: hindari pemicu yang anda................................................

REF : Unduh dan pasang aplikasi BlockLauncher. Kunjungi situs mod Minecraft PE. Carilah mod Herobrine. Ketuk tautan "Download Script" di bagian bawah laman. Ketuk tautan "Download Texture Pack". Jalankan gi
PRED: unduh berkas..........................................................


## **Fine-tuning model Pre-Train**

Default `google/mt5-small`. Model lain yang kompatibel dengan `AutoModelForSeq2SeqLM` (mis. model Indonesia) bisa dipakai setelah kamu **memeriksa ketersediaannya di Hugging Face**; cukup ganti `PRETRAINED` di sel konfigurasi.

Catatan: mT5 dapat menghasilkan loss `NaN` jika dilatih dengan **fp16**, jadi notebook ini memakai fp32 (batch kecil + gradient accumulation).

**Load model dan tokenizer pretrained**, serta fungsi `pt_collate` yang memotong input ke `max_src_tok` dan target ke `max_tgt_tok` (token padding pada label diganti -100 agar tidak dihitung dalam loss).

In [ ]:
pcfg = PT_CFG
tok = AutoTokenizer.from_pretrained(PRETRAINED)
pt_model = AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED).to(DEVICE)

def pt_collate(batch):
    docs = [b["document"] for b in batch]
    sums = [b["summary"] for b in batch]
    enc = tok(docs, max_length=pcfg["max_src_tok"], truncation=True, padding=True, return_tensors="pt")
    lab = tok(text_target=sums, max_length=pcfg["max_tgt_tok"], truncation=True, padding=True, return_tensors="pt")["input_ids"]
    lab[lab == tok.pad_token_id] = -100
    enc["labels"] = lab
    return enc

pt_train_dl = DataLoader(train_rows, batch_size=pcfg["batch_size"], shuffle=True, collate_fn=pt_collate)
pt_val_dl = DataLoader(val_rows, batch_size=pcfg["eval_batch_size"], shuffle=False, collate_fn=pt_collate)
print("Parameter:", sum(p.numel() for p in pt_model.parameters()) / 1e6, "juta")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Parameter: 300.176768 juta


**Fine-tuning.** Loop training manual dalam fp32 dengan gradient accumulation (batch efektif = `batch_size` x `grad_accum`), scheduler linear dengan warmup, dan penyimpanan checkpoint terbaik (val loss) ke Drive. Jika checkpoint sudah ada dan `RETRAIN = False`, training dilewati.

In [ ]:
def pt_eval_loss(model, dl):
    model.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            tot += model(**b).loss.item(); n += 1
    return tot / max(n, 1)

pt_dir = f"{CKPT_DIR}/{PRETRAINED.split('/')[-1]}_best"
pt_meta = f"{CKPT_DIR}/pt_meta.json"

if os.path.exists(f"{pt_dir}/config.json") and not RETRAIN:
    print("Checkpoint ditemukan, training dilewati:", pt_dir)
    pt_train_time = json.load(open(pt_meta))["train_time_s"] if os.path.exists(pt_meta) else None
else:
    opt = torch.optim.AdamW(pt_model.parameters(), lr=pcfg["lr"])
    steps = (len(pt_train_dl) // pcfg["grad_accum"] + 1) * pcfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    best, pt_history = 1e9, []
    t0 = time.time()
    for ep in range(1, pcfg["epochs"] + 1):
        pt_model.train()
        tot, n = 0.0, 0
        opt.zero_grad()
        for i, b in enumerate(pt_train_dl, 1):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = pt_model(**b).loss / pcfg["grad_accum"]
            loss.backward()
            tot += loss.item() * pcfg["grad_accum"]; n += 1
            if i % pcfg["grad_accum"] == 0 or i == len(pt_train_dl):
                torch.nn.utils.clip_grad_norm_(pt_model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
        va = pt_eval_loss(pt_model, pt_val_dl)
        pt_history.append((ep, tot / n, va))
        print(f"Epoch {ep} | train loss {tot/n:.4f} | val loss {va:.4f}")
        if va < best:
            best = va
            pt_model.save_pretrained(pt_dir); tok.save_pretrained(pt_dir)   # checkpoint ke Drive
    pt_train_time = time.time() - t0
    json.dump({"train_time_s": round(pt_train_time, 1), "history": pt_history}, open(pt_meta, "w"))
    print(f"Waktu training {PRETRAINED}: {pt_train_time:.1f} detik")
pt_model = AutoModelForSeq2SeqLM.from_pretrained(pt_dir).to(DEVICE)

Checkpoint ditemukan, training dilewati: /content/drive/MyDrive/dokumen_nlp/checkpoints/mt5-small_best


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


**Inferensi.** Generate ringkasan dengan beam search (`num_beams`), `no_repeat_ngram_size=3` untuk mengurangi pengulangan, lalu simpan hasil ke JSONL.

In [ ]:
@torch.no_grad()
def pt_predict(model, rows):
    model.eval()
    preds = []
    for i in range(0, len(rows), pcfg["eval_batch_size"]):
        docs = [r["document"] for r in rows[i:i + pcfg["eval_batch_size"]]]
        enc = tok(docs, max_length=pcfg["max_src_tok"], truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        out = model.generate(**enc, max_new_tokens=pcfg["max_tgt_tok"], num_beams=pcfg["num_beams"],
                             no_repeat_ngram_size=3, early_stopping=True)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

t0 = time.time()
pt_preds = pt_predict(pt_model, test_rows)
pt_infer_time = time.time() - t0
name = PRETRAINED.split("/")[-1].replace("-", "_")
save_predictions(f"{OUT_DIR}/abstractive_{name}.jsonl", test_rows, pt_preds)
print(f"Waktu inferensi: {pt_infer_time:.1f} detik untuk {len(test_rows)} contoh")
for i in range(3):
    print("\nREF :", test_rows[i]["summary"][:200]); print("PRED:", pt_preds[i][:200])

## **Checking ROUGE dan catat hasil running**

Pada tahap ini hanya melakukan pengecekan sementara. **Skor dihitung di tahhap 04_evaluation** dengan skrip bersama.

In [ ]:
def quick_rouge(preds, refs):
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"])
    res = {k: [] for k in ("rouge1", "rouge2", "rougeL")}
    for p, r in zip(preds, refs):
        s = sc.score(r, p)
        for k in res: res[k].append(s[k].fmeasure)
    return {k: round(float(np.mean(v)), 4) for k, v in res.items()}

refs = [r["summary"] for r in test_rows]
print("GRU+attention :", quick_rouge(lstm_preds, refs))
print(PRETRAINED, ":", quick_rouge(pt_preds, refs))

run_info = {
    "debug": DEBUG, "n_train": len(train_rows), "n_val": len(val_rows), "n_test": len(test_rows),
    "lstm": {"cfg": LSTM_CFG, "train_time_s": lstm_train_time, "infer_time_s": round(lstm_infer_time, 1)},
    "pretrained": {"model": PRETRAINED, "cfg": PT_CFG, "train_time_s": pt_train_time, "infer_time_s": round(pt_infer_time, 1)},
}
with open(f"{OUT_DIR}/abstractive_run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
print("Catatan run disimpan di", f"{OUT_DIR}/abstractive_run_info.json")

## **Hasil Setelah Running**
- Jika `DEBUG = False`: unduh `abstractive_*.jsonl` dan `abstractive_run_info.json` dari `dokumen_nlp/results/` di Drive, lalu unggah ke folder `results/` di branch `abstractive` GitHub.
- Jika `DEBUG = True`: hasilnya hanya uji coba (folder `scratch/`), tidak perlu diunggah ke `results/`. Setelah semua sel lolos, ubah `DEBUG = False` dan jalankan ulang (Run all).
- Checkpoint model (`checkpoints/`) tetap di Drive dan tidak dimasukkan ke GitHub.